### To know Model UPDRS progression over time, not just point predictions

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
data = pd.read_csv("C:/Users/Durosimi/PROJECTS/Parkinson/data/processed/features_df.csv")

In [5]:
data = data.sort_values(["subject", "test_time"]).reset_index(drop=True)

In [7]:
data.head()

,subject,test_time,subject.1,test_time.1,motor_updrs,jitter,shimmer,hnr,dfa,rpde,...,log_ppe,log_hnr,dysphonia_index,age_jitter_interaction,age_shimmer_interaction,delta_lag_1,time_diff,delta_trend_3,baseline_updrs,delta_motor_updrs
0,1,5.6431,1,5.6431,28.199,0.00348,0.01192,27.927,0.52499,0.37340,...,0.157568,3.364775,-0.346134,0.25056,0.85824,8.778,-169.0169,5.518333,28.199,0.0
1,1,5.6438,1,5.6438,28.199,0.00413,0.01282,26.641,0.53637,0.50911,...,0.224606,3.319300,0.228870,0.29736,0.92304,8.778,-169.0162,5.674000,28.199,0.0
2,1,5.6451,1,5.6451,28.199,0.00217,0.00798,30.749,0.54572,0.41216,...,0.090484,3.457861,-0.905966,0.15624,0.57456,8.778,-169.0149,5.674000,28.199,0.0
3,1,5.6458,1,5.6458,28.199,0.00294,0.01058,27.530,0.53572,0.40986,...,0.153879,3.350956,-0.471970,0.21168,0.76176,8.779,-169.0142,5.674333,28.199,0.0
4,1,5.6458,1,5.6458,28.199,0.00250,0.00631,32.102,0.52767,0.31053,...,0.063854,3.499594,-0.908366,0.18000,0.45432,8.779,-169.0142,5.674333,28.199,0.0


In [36]:
data.columns

Index(['subject', 'test_time', 'subject.1', 'test_time.1', 'motor_updrs',
       'jitter', 'shimmer', 'hnr', 'dfa', 'rpde', 'ppe', 'age', 'sex',
       'voice_instability_mean', 'voice_instability_std',
       'jitter_shimmer_ratio', 'log_ppe', 'log_hnr', 'dysphonia_index',
       'age_jitter_interaction', 'age_shimmer_interaction', 'delta_lag_1',
       'time_diff', 'delta_trend_3', 'baseline_updrs', 'delta_motor_updrs',
       'updrs_lag1', 'updrs_lag2', 'updrs_delta', 'updrs_roll3',
       'updrs_roll10', 'delta_lag1'],
      dtype='object')

### Creating lag features

In [9]:
data["updrs_lag1"] = data.groupby("subject")["motor_updrs"].shift(1)
data["updrs_lag2"] = data.groupby("subject")["motor_updrs"].shift(2)

In [12]:
data["updrs_delta"] = data["motor_updrs"] - data["updrs_lag1"]

### Check for rolling trend

In [14]:
data["updrs_roll3"] = (data.groupby("subject")["motor_updrs"].rolling(window=3, min_periods=1).mean().reset_index(level=0, drop=True))


In [38]:
data["updrs_roll10"] = data.groupby("subject")["motor_updrs"].rolling(window=10, min_periods=1).mean().reset_index(level=0, drop=True)
data["delta_lag1"] = data["updrs_lag1"] - data["updrs_lag2"]

In [43]:
data["updrs_from_baseline"] = data["motor_updrs"] - data["baseline_updrs"]

### Drop null values

In [45]:
data = data.dropna().reset_index(drop=True)

In [47]:
target = "updrs_delta"

features = [
    "updrs_lag1",
    "updrs_lag2",
    "updrs_roll10",
    "updrs_roll3",
    "delta_lag1",
    "updrs_from_baseline",
    "jitter",
    "shimmer",
    "hnr",
    "rpde",
    "ppe",
    "sex"
]

In [49]:
X = data[features]
y = data[target]
subjects = data["subject"]

split_idx = int(len(data) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
subjects_train = subjects.iloc[:split_idx]


In [51]:
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
     ("ridge", Ridge(alpha=0.1))
      ]
)

In [53]:
rmse_scores = []
mae_scores = []
cv = GroupKFold(n_splits=5)

for train_idx, val_idx in cv.split(X_train, y_train, groups=subjects_train):
    ridge_pipeline.fit(
        X_train.iloc[train_idx],
        y_train.iloc[train_idx]
    )

    preds = ridge_pipeline.predict(X_train.iloc[val_idx])

    rmse_scores.append(
        np.sqrt(mean_squared_error(y_train.iloc[val_idx], preds))
    )
    mae_scores.append(
        mean_absolute_error(y_train.iloc[val_idx], preds)
    )

print("CV MAE:", np.mean(mae_scores))
print("CV RMSE:", np.mean(rmse_scores))

CV MAE: 0.02958349247646571
CV RMSE: 0.06836879477792536


In [55]:
ridge_pipeline.fit(X_train, y_train)

y_pred = ridge_pipeline.predict(X_test)

print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("Test R2:", r2_score(y_test, y_pred))


Test MAE: 0.024787528426530196
Test RMSE: 0.046080372405029636
Test R2: 0.9283741658080131


### We demonstrate that modeling Parkinson’s progression requires temporal anchoring and patient-specific baselines. When formulated as a delta prediction problem and evaluated using group-aware temporal cross-validation, a simple regularized linear model outperforms complex baselines while remaining interpretable and clinically meaningful.”

In [19]:
type(cv)

sklearn.model_selection._split.GroupKFold

In [20]:
splits = list(cv.split(X_train, y_train, groups=subjects_train))
len(splits), type(splits[0]), len(splits[0])


(5, tuple, 2)

In [21]:
print("cv repr:", cv)

cv repr: GroupKFold(n_splits=5, random_state=None, shuffle=False)


In [22]:
gen = cv.split(X_train, y_train, groups=subjects_train)

first = next(gen)
print("Type of one split:", type(first))
print("Length of split:", len(first))
print("Split content types:", [type(x) for x in first])


Type of one split: <class 'tuple'>
Length of split: 2
Split content types: [<class 'numpy.ndarray'>, <class 'numpy.ndarray'>]


In [23]:
print(type(subjects_train))
print(len(subjects_train), len(X_train))
print(subjects_train[:5])

<class 'pandas.core.series.Series'>
4599 4599
0    1
1    1
2    1
3    1
4    1
Name: subject, dtype: int64


In [61]:
data["delta_std_rolling"] = (
    data.groupby("subject")["updrs_delta"]
      .rolling(5)
      .std()
      .reset_index(level=0, drop=True)
)


In [63]:
patient_stats = (
    data.groupby("subject")
      .agg(
          mean_delta=("updrs_delta", "mean"),
          std_delta=("updrs_delta", "std"),
          baseline=("baseline_updrs", "first"),
          visits=("subject", "count")
      )
)


In [71]:
patient_stats.head()

,mean_delta,std_delta,baseline,visits
subject,,,,
1,0.060130,0.143554,28.199,146
2,0.054514,0.159598,11.078,142
3,0.032440,0.126465,23.437,141
4,0.059948,0.163382,10.737,134
5,0.018046,0.065021,31.000,153


### Prediction errors are dominated by patient-specific disease volatility rather than missing temporal information, highlighting intrinsic unpredictability in Parkinson’s progression.

### To remove motor_updrs after c=testing temporal_error_analysis copy

In [57]:
final_columns = [
    "baseline_updrs",
    "subject",
    "motor_updrs",
    "test_time",
    "updrs_lag1",
    "updrs_lag2",
    "updrs_roll10",
    "delta_lag1",
    "updrs_from_baseline",
    "delta_motor_updrs",
    "updrs_roll3",
    "jitter",
    "shimmer",
    "hnr",
    "rpde",
    "ppe",
    "sex"
]
features_temporal_new = data[final_columns].copy()
features_temporal_new.to_csv("C:/Users/Durosimi/PROJECTS/Parkinson/data/processed/features_temporal_new.csv", index=False)


In [64]:
# final_model = Pipeline(
#     [
#         ("scaler", StandardScaler()),
#         ("ridge", Ridge(alpha=0.1))
#     ]
# )

# final_model.fit(X, y)

In [68]:
# y_pred = final_model.predict(X_test)

# print("Test MAE:", mean_absolute_error(y_test, y_pred))
# print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
# print("Test R2:", r2_score(y_test, y_pred))

In [59]:
joblib.dump(ridge_pipeline, "C:/Users/Durosimi/PROJECTS/Parkinson/models/ridge_temporal.pkl")

metadata = {
    "model": "Ridge",
    "alpha": 0.1,
    "features": features,
    "target": "delta_motor_updrs",
    "cv": "GroupKFold by subject",
    "trained_on": str(datetime.now())
}

with open("../models/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Model and metadata saved")

Model and metadata saved


In [74]:
data[final_columns].isna().mean()


baseline_updrs         0.0
subject                0.0
motor_updrs            0.0
test_time              0.0
updrs_lag1             0.0
updrs_lag2             0.0
updrs_roll10           0.0
delta_lag1             0.0
updrs_from_baseline    0.0
delta_motor_updrs      0.0
updrs_roll3            0.0
jitter                 0.0
shimmer                0.0
hnr                    0.0
rpde                   0.0
ppe                    0.0
sex                    0.0
dtype: float64